# 🎬 AI Video Assistant - Google Colab Edition
Run your AI YouTube Shorts Generator for FREE using Google's T4 GPU.

---

### 1️⃣ Setup Dependencies & Clone Repo
**Run this ONCE** at the start of your session. You do NOT need to run this again for subsequent videos.

**Note:** If your GitHub repository is **Private**, provide your **Personal Access Token (PAT)** below. If it is **Public**, leave it empty.

In [ ]:
import os

# @title GitHub Configuration
GITHUB_USERNAME = "hrishabht5" # @param {type:"string"}
GITHUB_REPO = "AI-video-Assistent" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}

!apt-get update
!apt-get install -y ffmpeg libavcodec-dev libavformat-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev imagemagick

# Fix ImageMagick policy for subtitles
!sed -i 's/rights="none" pattern="@\*"/rights="read|write" pattern="@*"/' /etc/ImageMagick-6/policy.xml

# 🛑 CRITICAL FIX for Python 3.12 (Colab's new default)
!pip install --upgrade pip setuptools

# Construct the Clone URL
if GITHUB_TOKEN:
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

# Clone the repository if it doesn't exist
if not os.path.exists(GITHUB_REPO):
    !git clone {clone_url}
else:
    print(f"Directory {GITHUB_REPO} already exists. Pulling latest changes.")
    %cd {GITHUB_REPO}
    !git pull
    %cd ..

%cd {GITHUB_REPO}
!pip install -r requirements.txt

# Install localtunnel for API access
!npm install -g localtunnel

### 2️⃣ Configure Gemini API Key
**Run this ONCE** at the start of your session.

In [ ]:
import os

GEMINI_API_KEY = "" # @param {type:"string"}
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY={GEMINI_API_KEY}')

### 🆕 OPTIONAL: Start the Web API
If you want to use the tool via a web interface/API instead of manual cells, run this cell.
It will give you a **public URL**.

In [ ]:
import threading
import time
import requests

# Run FastAPI in the background
def run_api():
    !python api.py

threading.Thread(target=run_api, daemon=True).start()
time.sleep(5)  # Wait for server to start

print("API started in background. Fetching public URL...")
print("IP Address for localtunnel (Copy this if prompted):", requests.get('https://loca.lt/mytunnelpassword').text)

# Expose port 8000 via localtunnel
!lt --port 8000

### 3️⃣ Run Batch Processing (CLI Mode)
Enter one or more YouTube URLs (comma separated) to process them sequentially.
**You can run this cell multiple times for new videos!**

In [ ]:
VIDEO_URLS = "" # @param {type:"string"}

# Split by comma or newline and strip whitespace
urls = [x.strip() for x in VIDEO_URLS.replace('\n', ',').split(',') if x.strip()]

if not urls:
    print("Please enter at least one URL.")
else:
    print(f"Found {len(urls)} videos to process.")
    for i, url in enumerate(urls):
        print(f"\n{'='*20} Processing Video {i+1}/{len(urls)} {'='*20}")
        print(f"URL: {url}")
        !python main.py --auto-approve "{url}"
        print(f"{'='*20} Finished Video {i+1} {'='*20}\n")

### 4️⃣ Save to Drive & Download
Run this to automatically save videos to your generated Google Drive folder.
**Tip:** Use `CLEANUP_AFTER_UPLOAD` to prevent re-uploading the same videos if you run this multiple times.

In [ ]:
from google.colab import drive
from google.colab import files
import shutil
import os

# @title Save Options
SAVE_TO_DRIVE = True # @param {type:"boolean"}
DRIVE_FOLDER = "AI_Shorts_Output" # @param {type:"string"}
CLEANUP_AFTER_UPLOAD = True # @param {type:"boolean"}
DOWNLOAD_LOCALLY = False # @param {type:"boolean"}

# Find generated files
mp4_files = [f for f in os.listdir('.') if f.endswith('.mp4') and '_short.mp4' in f]

if not mp4_files:
    print("No processed videos found to save. (They may have already been cleaned up)")
else:
    if SAVE_TO_DRIVE:
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        
        DESTINATION_PATH = f"/content/drive/My Drive/{DRIVE_FOLDER}"
        if not os.path.exists(DESTINATION_PATH):
            os.makedirs(DESTINATION_PATH)
            print(f"Created Google Drive folder: {DESTINATION_PATH}")
        
        print(f"Uploading {len(mp4_files)} videos to Drive folder: {DRIVE_FOLDER}...")
        for video in mp4_files:
            try:
                shutil.copy2(video, os.path.join(DESTINATION_PATH, video))
                print(f"✓ Saved to Drive: {video}")
                
                if CLEANUP_AFTER_UPLOAD:
                    os.remove(video)
                    print(f"  ↳ Removed local copy: {video}")
            except Exception as e:
                print(f"✗ Failed to save {video}: {str(e)}")

    if DOWNLOAD_LOCALLY and mp4_files:
        # Note: If cleanup is ON, we might have deleted them already, so this logic is tricky.
        # We'll zip before cleanup if both are selected, but for now assuming user picks one or the other mainly.
        # Or specific check:
        remaining_files = [f for f in mp4_files if os.path.exists(f)]
        if remaining_files:
            print("Zipping and downloading locally...")
            !zip -r shorts_output.zip *.mp4
            files.download("shorts_output.zip")
        else:
            print("Files were cleaned up, nothing to zip for local download.")